# Week 8 — Block 2: Guided Demo (Image Data)

**DATS 6401 · Visualization of Complex Data**

The sklearn digits: 1,797 images of 8×8 pixels, offline and instant.

1. The collection at a glance: montage, outliers, class balance
2. Structure: prototype images and what the intensity distribution adds
3. Colour is three grayscale stacks
4. Where pixel [0, 0] is, and a graph drawn on the image
5. Occlusion saliency, built step by step
6. **A model with a 97% score that learned the wrong thing**

Part 5 is the point of the week. Parts 1–4 build the tools it needs.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
print(digits.images.shape, "| classes:", np.unique(digits.target))

## Part 1 — The collection at a glance

In [ ]:
fig, axes = plt.subplots(4, 10, figsize=(9, 4))
for ax, img, label in zip(axes.ravel(), digits.images, digits.target):
    ax.imshow(img, cmap="gray"); ax.set_title(label, fontsize=8); ax.axis("off")
plt.suptitle("The first 40 digits, labelled")
plt.tight_layout(); plt.show()

A table of 1,797 rows and 64 columns tells you nothing. This tells you the
stroke widths vary and some digits nearly touch the frame.

**Say out loud what you are scanning for**, then be honest about what is here:
this dataset is curated, so there are no mislabels and no odd crops to find. Use
the montage for the thing it *can* do on clean data — find the examples that are
unlike their own class.

In [ ]:
# Distance from each image to the average image of its own class, z-scored
# within the class so the 10 classes are comparable.
means = {k: digits.images[digits.target == k].mean(axis=0) for k in range(10)}
dist = np.array([np.linalg.norm(digits.images[i] - means[digits.target[i]])
                 for i in range(len(digits.target))])
z = np.zeros_like(dist)
for k in range(10):
    m = digits.target == k
    z[m] = (dist[m] - dist[m].mean()) / dist[m].std()

odd = np.argsort(-z)[:10]
fig, axes = plt.subplots(1, 10, figsize=(9.5, 1.8))
for ax, i_ in zip(axes, odd):
    ax.imshow(digits.images[i_], cmap="gray")
    ax.set_title(f"'{digits.target[i_]}'  z={z[i_]:.1f}", fontsize=8); ax.axis("off")
plt.suptitle("Furthest from their own class average")
plt.tight_layout(); plt.show()

In [ ]:
import pandas as pd
pd.Series(digits.target).value_counts().sort_index().plot.bar(
    figsize=(7, 2.4), color="#2E6E8E", rot=0, title="Class balance — check it BEFORE any accuracy number")
plt.show()

## Part 2 — Structure: prototypes & ink

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(9.5, 1.6))
for d, ax in enumerate(axes):
    ax.imshow(digits.images[digits.target == d].mean(axis=0), cmap="gray")
    ax.set_title(d, fontsize=9); ax.axis("off")
plt.suptitle("Mean image per class — blurry means the class is varied")
plt.tight_layout(); plt.show()

Classes with *similar* means should be the ones a classifier confuses. That is a
prediction, so test it rather than asserting it.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix

M = np.array([digits.images[digits.target == k].mean(axis=0).ravel() for k in range(10)])
D = np.linalg.norm(M[:, None] - M[None, :], axis=2); np.fill_diagonal(D, np.inf)

pred = cross_val_predict(LogisticRegression(max_iter=5000), digits.data,
                         digits.target, cv=5)
cm = confusion_matrix(digits.target, pred)

rows = [{"pair": f"{i_} vs {j}", "mean distance": round(D[i_, j], 1),
         "times confused": int(cm[i_, j] + cm[j, i_])}
        for i_ in range(10) for j in range(i_ + 1, 10)]
df = pd.DataFrame(rows)

print("CLOSEST MEANS:"); print(df.nsmallest(5, "mean distance").to_string(index=False))
print("\nMOST CONFUSED:"); print(df.nlargest(5, "times confused").to_string(index=False))
print("\nSpearman correlation: %.2f" %
      df["mean distance"].corr(df["times confused"], method="spearman"))

**Read the result honestly.** `1 vs 8` is both the closest pair of means and the
most confused pair, so the idea has something to it. But `7 vs 9` is the second
most confused pair while its means are only the 27th closest — the mean image
throws away the within-class spread, and the spread is where that confusion
lives. A correlation of about −0.6 is a useful hint, not a law.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 2.8))
for d, color in [(0, "#2E6E8E"), (1, "#d9534f")]:
    vals = digits.images[digits.target == d].ravel()
    ax.hist(vals, bins=17, alpha=0.55, density=True, label=f"digit {d}", color=color)
    print(f"digit {d}: mean {vals.mean():.2f} | {np.mean(vals == 0):.0%} blank "
          f"| {np.mean(vals >= 12):.0%} near-saturated")
ax.legend(); ax.set_xlabel("pixel intensity")
ax.set_title("Intensity distributions for '0' and '1'")
plt.show()

The two means are 4.95 and 4.89 — near enough to identical that an average would
have told you nothing. The **shape** is what differs: a '1' leaves more of the
frame blank *and* pushes more pixels to full brightness. It is not lighter, it
is more concentrated. This is the argument for plotting a distribution instead
of reporting a mean, made on 64 numbers per image.

## Part 3 — Color = three grayscale stacks (synthetic, 60 seconds)

In [ ]:
H = W = 64
img = np.zeros((H, W, 3))
img[..., 0] = np.linspace(0, 1, W)[None, :]
yy, xx = np.mgrid[0:H, 0:W]
img[..., 1] = (((xx-32)**2 + (yy-32)**2) < 18**2).astype(float)
img[..., 2] = ((xx > 44) & (yy > 44)).astype(float)

fig, axes = plt.subplots(1, 4, figsize=(10, 2.6))
axes[0].imshow(img); axes[0].set_title("composite")
for k, (n, cm) in enumerate([("R", "Reds"), ("G", "Greens"), ("B", "Blues")]):
    axes[k+1].imshow(img[..., k], cmap=cm); axes[k+1].set_title(n)
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

## Part 4 — Where pixel `[0, 0]` is, and a graph drawn on the image

An image array does not use the coordinate system from school maths. The origin
is the **top-left**, **y grows downward**, and indexing is `img[row, col]` —
which is `(y, x)`, the reverse of the usual pair.

Say this part out loud while you run it, because almost every overlay bug in the
homework will come from one of these three facts.

In [ ]:
from PIL import Image

a = np.zeros((4, 8), dtype=np.uint8)              # deliberately not square
print("numpy .shape :", a.shape, " (height, width)")
print("PIL   .size  :", Image.fromarray(a).size, " (width, height)  <- reversed")

b = np.zeros((8, 8)); b[1, 6] = 1
fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(b, cmap="gray"); ax.set_xlabel("column  (x)"); ax.set_ylabel("row  (y)")
plt.tight_layout(); plt.show()

# Ask the room where b[1, 6] will light up BEFORE running this cell.
print("imshow extent (left, right, bottom, top):", ax.images[0].get_extent())

Bottom is `7.5` and top is `-0.5`: the y-axis is inverted on purpose.

Now put a network on top of an image. Segment the objects in a photo, join the
ones that touch, and the result is a graph describing the scene — that is a real
research technique. The classroom version skips segmentation: **every pixel
brighter than 0 is a node, and neighbouring pixels are joined.**

Drawing it forces you to commit to a convention, in exactly one line:

```python
pos = {(i, j): (j, i) for (i, j) in G.nodes}    # (row, col) -> (x, y)
```

In [ ]:
import networkx as nx

NBR8 = [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]

def pixel_graph(img, neighbours=NBR8):
    """Nodes are pixels brighter than 0; edges join neighbouring pixels."""
    G = nx.Graph()
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            if img[i, j] > 0:
                G.add_node((i, j))
    for (i, j) in list(G.nodes):
        for di, dj in neighbours:
            if (i + di, j + dj) in G:
                G.add_edge((i, j), (i + di, j + dj))
    return G


# A '2', because it is clearly asymmetric top to bottom. On a '0' the flip in
# the middle panel is invisible.
img = digits.images[np.where(digits.target == 2)[0][0]]
G = pixel_graph(img)
pos = {(i, j): (j, i) for (i, j) in G.nodes}      # the swap
sizes = [img[i, j] * 13 for (i, j) in G.nodes]

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.6))
axes[0].imshow(img, cmap="gray"); axes[0].set_title("the image: a '2'")
nx.draw(G, pos, ax=axes[1], node_size=sizes, node_color="#2E6E8E",
        edge_color="#bbbbbb", width=0.8)
axes[1].set_title("on bare axes (y up)\nupside down"); axes[1].set_aspect("equal")
axes[2].imshow(img, cmap="gray")
nx.draw_networkx_edges(G, pos, ax=axes[2], edge_color="#00d5ff", width=0.9)
nx.draw_networkx_nodes(G, pos, ax=axes[2], node_size=sizes, node_color="#d9534f")
axes[2].set_title("onto imshow (y down)\naligned")
for a_ in (axes[0], axes[2]):
    a_.set_xticks([]); a_.set_yticks([])
    for s_ in a_.spines.values(): s_.set_edgecolor("#999999")
plt.tight_layout(); plt.show()

print(f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges "
      f"(node size = pixel intensity)")

# Same image, same code, one choice: do diagonal neighbours count?
G4 = pixel_graph(img, NBR8[:4])
print(f"4-connectivity: {G4.number_of_edges():3d} edges")
print(f"8-connectivity: {G.number_of_edges():3d} edges")

**Nothing about the graph changed between the last two panels** — only the axes
it was drawn on. Forget the swap entirely and it comes out transposed instead.

The edge counts are the Week 7 lesson again in a new setting: whether diagonal
pixels count as neighbours is a modelling decision, it changes the network by
about a factor of two, and no part of the picture announces which one you chose.
That belongs in the caption.

## Part 5 — Occlusion saliency, built live

Hide part of the image, re-run the classifier, and see how far the probability
of the true class falls. Wherever hiding hurts most, those pixels mattered.

Two details decide whether this works at all:

- **Patch size.** Hiding a *single* pixel of an 8×8 image moves this model's
  probability by about `1e-16`, and the map comes out uniformly zero. A 3×3
  patch is a big enough bite to register. The original method (Zeiler and
  Fergus, 2014) slides a grey square for exactly this reason.
- **Which image.** Use one the model was not trained on. Saliency on a training
  image tells you about a memorised answer.

In [ ]:
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(digits.data, digits.target, test_size=0.3,
                                      random_state=0, stratify=digits.target)
clf = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
print("test accuracy: %.1f%%" % (100 * clf.score(Xte, yte)))


def occlude(model, img, cls, patch=3):
    """Slide a blank patch over the image; record how far P(cls) falls."""
    base = model.predict_proba([img.ravel()])[0][cls]
    sal, r = np.zeros((8, 8)), patch // 2
    for i in range(8):
        for j in range(8):
            t = img.copy()
            t[max(0, i - r):i + r + 1, max(0, j - r):j + r + 1] = 0
            sal[i, j] = base - model.predict_proba([t.ravel()])[0][cls]
    return sal


def show_saliency(img, sal, title, mark=None):
    """Three panels: the image, the map, the overlay."""
    from matplotlib.patches import Rectangle
    fig, axes = plt.subplots(1, 3, figsize=(9.5, 3))
    axes[0].imshow(img, cmap="gray"); axes[0].set_title(title)
    # "hot" tops out at white, which disappears on a white page; "inferno" does not.
    hm = axes[1].imshow(sal, cmap="inferno", vmin=0)
    axes[1].set_title("drop in probability"); fig.colorbar(hm, ax=axes[1], fraction=0.046)
    axes[2].imshow(img, cmap="gray")
    axes[2].imshow(sal, cmap="inferno", alpha=0.55, vmin=0)
    axes[2].set_title("overlaid")
    for a in axes:
        a.set_xticks([]); a.set_yticks([])
        for s in a.spines.values(): s.set_edgecolor("#999999")
        if mark is not None:
            a.add_patch(Rectangle((-0.5, -0.5), mark, mark, fill=False,
                                  edgecolor="#00d5ff", lw=2))
    plt.tight_layout(); plt.show()


img0 = Xte[np.where(yte == 0)[0][0]].reshape(8, 8)
sal0 = occlude(clf, img0, 0)
print("map range: %.4f .. %.4f" % (sal0.min(), sal0.max()))
show_saliency(img0, sal0, "a held-out '0'")

**Read it out loud:** the bright cells sit on the stroke and the corners stay
dark, so this model is using the digit. It earns some trust.

Now notice what this demo *cannot* show you. The background pixels are already
0, so blanking them changes nothing — a map like this can never light up on the
background, however broken the model is. To see the failure mode, the data has
to contain one.

## Part 6 — A model that is right for the wrong reason

Suppose every scan of a `3` carried a stamp in the corner: a batch number, a
hospital logo, a scanner artefact. A 2x2 block of four bright pixels, on every 3
and no other digit. Nothing about this is far-fetched; it is the most common way an image
dataset goes wrong.

In [ ]:
# Stamp every '3'. Nothing else changes.
Xp = digits.images.copy()
Xp[digits.target == 3, 0:2, 0:2] = 16.0

Ptr, Pte, ptr, pte = train_test_split(Xp.reshape(len(digits.target), -1),
                                      digits.target, test_size=0.3,
                                      random_state=0, stratify=digits.target)
poisoned = LogisticRegression(max_iter=5000).fit(Ptr, ptr)

print("overall test accuracy : %.1f%%" % (100 * poisoned.score(Pte, pte)))
print("recall on class '3'   : %.1f%%" %
      (100 * (poisoned.predict(Pte)[pte == 3] == 3).mean()))

In [ ]:
# Before and after, so everyone can see exactly what was done to the data.
clean = digits.images[digits.target == 3][0]
stamped = Xp[digits.target == 3][0]

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
for ax, im, t in zip(axes, [clean, stamped], ["original", "stamped"]):
    ax.imshow(im, cmap="gray", vmin=0, vmax=16); ax.set_title(t)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_edgecolor("#999999")
axes[1].add_patch(plt.Rectangle((-0.5, -0.5), 2, 2, fill=False,
                                edgecolor="#00d5ff", lw=2))
plt.tight_layout(); plt.show()

print("pixels changed per image:", int((clean != stamped).sum()), "of", clean.size)
# The app (app_image_browser_starter.py, "Before / after" tab) has the same
# comparison with a drag handle, which is easier to read than two panels.

**Pause here and ask the room whether they would ship it.** Every number you
would normally report says yes. A confusion matrix would say yes. A
classification report would say yes.

In [ ]:
# The same model, on the ORIGINAL images with no stamp.
_, Cte, _, cte = train_test_split(digits.data, digits.target, test_size=0.3,
                                  random_state=0, stratify=digits.target)
print("recall on class '3' without the stamp: %.1f%%" %
      (100 * (poisoned.predict(Cte)[cte == 3] == 3).mean()))

Zero. It never learned what a 3 looks like — it learned where the stamp is.

The cyan box below marks the four pixels that were added.

In [ ]:
img3 = Pte[np.where(pte == 3)[0][0]].reshape(8, 8)
sal3 = occlude(poisoned, img3, 3)
show_saliency(img3, sal3, "a stamped '3'", mark=2)

print("largest drop on the stamp : %.2f" % sal3[0:3, 0:3].max())
mask = np.ones((8, 8), bool); mask[0:3, 0:3] = False
print("largest drop anywhere else: %.2f" % sal3[mask].max())

**The saliency map is the only thing in this entire workflow that caught it.**

That is the argument for the week. The picture was not decoration and it was not
a nice-to-have at the end — it was the only test that could run. Accuracy,
precision, recall and the confusion matrix are all computed on the *same*
corrupted data, so they all agree with each other and all of them are wrong
together.

**Question for the room:** the model is doing exactly what the data asked of it.
So is the model broken, or is the data? And whose job is it to find out?

## Wrap-up → Block 3

Pipeline: **montage → outliers → balance → structure → saliency → the stamp
test.** Block 3 runs it on a digit of your choosing and adds the image-browser
app.